# Engineered Features Comparison: Train.csv vs Test.csv

This notebook compares the training and test datasets using engineered features (431 features) instead of raw features.
This analysis helps us understand what possible shifts the model is dealing with after feature engineering.

We will:
1. Load both datasets
2. Apply feature engineering to both datasets
3. Handle missing values (-9999 in test data)
4. Compare statistical distributions of engineered features
5. Perform statistical tests to assess similarity
6. Visualize differences

**Note**: The test dataset contains -9999 values which represent missing data and should be treated as NaN.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Add project root to path to import feature engineering module
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))
from aquaculture.feature_engineering import AquacultureFeatureEngineer

In [ ]:
# Load the datasets
print("Loading datasets...")
train_df = pd.read_csv('../data/Train.csv')
test_df = pd.read_csv('../data/Test.csv')

print(f"Training dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

In [ ]:
# Prepare raw data for feature engineering
# For training data: exclude ID and label columns
# For test data: exclude ID column

train_feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
test_feature_cols = [col for col in test_df.columns if col != 'ID']

X_train_raw = train_df[train_feature_cols].values
X_test_raw = test_df[test_feature_cols].values

# Reshape to 3D as expected by the feature engineer: (n_samples, 12, 12)
X_train = X_train_raw.reshape(X_train_raw.shape[0], 12, 12)
X_test = X_test_raw.reshape(X_test_raw.shape[0], 12, 12)

print(f"Reshaped training data shape: {X_train.shape}")
print(f"Reshaped test data shape: {X_test.shape}")

In [ ]:
# Create and fit feature engineer
print("Creating and fitting feature engineer...")
# For experimentation on the fly, we'll create a new feature engineer from scratch
# Modify these parameters to experiment with different feature engineering configurations

feature_engineer = AquacultureFeatureEngineer(
    simulate_mask=False,  # No stochastic masking for fair comparison
    random_state=42,
    include_optical=True,      # Set to False to exclude optical features
    include_sar=True,          # Set to False to exclude SAR features
    include_cross_sensor_features=True,  # Set to False to exclude cross-sensor features
    include_temporal_statistics=True,    # Set to False to exclude temporal statistics
    include_metadata=True      # Set to False to exclude metadata features
)

# Optional: Add feature selection for experimentation
# Uncomment and modify the following lines to experiment with feature selection
"""
from aquaculture.feature_selection import FeatureSelector

# Example 1: Select specific feature groups
# feature_selector = FeatureSelector(
#     feature_engineer,
#     selection_method='groups',
#     groups=['temporal', 'metadata']  # Only temporal and metadata features
# )

# Example 2: Select features by pattern
# feature_selector = FeatureSelector(
#     feature_engineer,
#     selection_method='patterns',
#     patterns=['_mean$', '_std$']  # Only mean and std features
# )

# Example 3: Select specific feature names
# feature_selector = FeatureSelector(
#     feature_engineer,
#     selection_method='names',
#     names=['VH_01', 'VV_01', 'green_01_mean', 'nir_01_std']  # Specific features
# )
"""

# For now, we'll use no feature selection (all features)
feature_selector = None
print("✓ Created new feature engineer (no feature selection for clean comparison)")

# Fit on training data to establish feature names
print("Fitting feature engineer on training data...")
try:
    feature_engineer.fit(X_train)
    print("✓ Fitted feature engineer on training data")
except Exception as e:
    print(f"⚠ Error fitting feature engineer: {e}")

# Transform both datasets
print("Transforming training data...")
try:
    if feature_selector is not None:
        # Use the full pipeline: feature engineering then feature selection
        X_train_engineered = feature_engineer.transform(X_train, training=False)
        X_train_features = feature_selector.transform(X_train_engineered, training=False)
        print(f"✓ Transformed training data: {X_train_engineered.shape[1]} → {X_train_features.shape[1]} features (with selection)")
    else:
        # Only feature engineering
        X_train_features = feature_engineer.transform(X_train, training=False)
        print(f"✓ Transformed training data: {X_train_features.shape[1]} features (no selection)")
except Exception as e:
    print(f"⚠ Error transforming training data: {e}")
    # Fallback
    X_train_features = feature_engineer.transform(X_train, training=False)

print("Transforming test data...")
try:
    if feature_selector is not None:
        # Use the same fitted transformer for test data
        X_test_engineered = feature_engineer.transform(X_test, training=False)
        X_test_features = feature_selector.transform(X_test_engineered, training=False)
        print(f"✓ Transformed test data: {X_test_engineered.shape[1]} → {X_test_features.shape[1]} features (with selection)")
    else:
        # Only feature engineering
        X_test_features = feature_engineer.transform(X_test, training=False)
        print(f"✓ Transformed test data: {X_test_features.shape[1]} features (no selection)")
except Exception as e:
    print(f"⚠ Error transforming test data: {e}")
    # Fallback
    X_test_features = feature_engineer.transform(X_test, training=False)

# Get feature names
try:
    if hasattr(X_train_features, 'columns'):
        feature_names = list(X_train_features.columns)
    elif hasattr(feature_engineer, 'get_feature_names_out'):
        # Try to get feature names after selection if possible
        try:
            if feature_selector is not None:
                feature_names = list(feature_selector.get_feature_names_out())
            else:
                feature_names = list(feature_engineer.get_feature_names_out())
        except:
            feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]
    else:
        feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]
except:
    feature_names = [f"feature_{i}" for i in range(X_train_features.shape[1])]

# Convert to DataFrames for easier analysis
try:
    train_features_df = pd.DataFrame(X_train_features.values, columns=feature_names)
    test_features_df = pd.DataFrame(X_test_features.values, columns=feature_names)
    print(f"Engineered features shape - Train: {train_features_df.shape}")
    print(f"Engineered features shape - Test: {test_features_df.shape}")
    print(f"Number of features: {len(feature_names)}")
    
    # Save the dataframes to CSV for further analysis
    train_features_df.to_csv('../data/engineered_train_features.csv', index=False)
    test_features_df.to_csv('../data/engineered_test_features.csv', index=False)
except Exception as e:
    print(f"⚠ Error creating DataFrames or saving CSV: {e}")

In [ ]:
# Check for missing values (-9999 in test data that became NaN after feature engineering)
print("Checking for missing values after feature engineering...")

train_missing = train_features_df.isnull().sum().sum()
test_missing = test_features_df.isnull().sum().sum()

print(f"Missing values in training features: {train_missing}")
print(f"Missing values in test features: {test_missing}")

# Show columns with missing values in test data (if any)
if test_missing > 0:
    missing_cols = test_features_df.columns[test_features_df.isnull().any()].tolist()
    print(f"\nColumns with missing values in test data ({len(missing_cols)} columns):")
    for i, col in enumerate(missing_cols[:10]):  # Show first 10
        missing_count = test_features_df[col].isnull().sum()
        print(f"  {col}: {missing_count} missing values")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
else:
    print("\n✓ No missing values found in test features")

## Basic Statistics Comparison

Let's compare basic statistics (mean, median, std) for each engineered feature between train and test datasets.

In [ ]:
# Calculate basic statistics
print("Calculating basic statistics...")

# Compute statistics for training data
train_stats = train_features_df.describe()

# Compute statistics for test data (ignoring NaN values)
test_stats = test_features_df.describe()

# Create a comparison dataframe
comparison_stats = pd.DataFrame()
comparison_stats['train_mean'] = train_stats.loc['mean']
comparison_stats['test_mean'] = test_stats.loc['mean']
comparison_stats['train_std'] = train_stats.loc['std']
comparison_stats['test_std'] = test_stats.loc['std']
comparison_stats['train_median'] = train_features_df.median()
comparison_stats['test_median'] = test_features_df.median()

# Calculate differences
comparison_stats['mean_diff'] = comparison_stats['test_mean'] - comparison_stats['train_mean']
comparison_stats['mean_diff_pct'] = (comparison_stats['mean_diff'] / comparison_stats['train_mean'].abs()) * 100
comparison_stats['std_ratio'] = comparison_stats['test_std'] / comparison_stats['train_std']

# Replace infinite values from division by zero
comparison_stats['mean_diff_pct'] = comparison_stats['mean_diff_pct'].replace([np.inf, -np.inf], np.nan)
comparison_stats['std_ratio'] = comparison_stats['std_ratio'].replace([np.inf, -np.inf], np.nan)

print("\nComparison statistics (first 10 features):")
display(comparison_stats.head(10))

# Summary of differences
print(f"\nMean difference statistics:")
print(f"  Mean absolute difference: {comparison_stats['mean_diff'].abs().mean():.4f}")
print(f"  Median absolute difference: {comparison_stats['mean_diff'].abs().median():.4f}")
print(f"  Max absolute difference: {comparison_stats['mean_diff'].abs().max():.4f}")

print(f"\nMean difference percentage statistics:")
print(f"  Mean absolute % difference: {np.nanmean(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")
print(f"  Median absolute % difference: {np.nanmedian(np.abs(comparison_stats['mean_diff_pct'])):.2f}%")

print(f"\nStd ratio statistics:")
print(f"  Mean std ratio: {np.nanmean(comparison_stats['std_ratio']):.4f}")
print(f"  Median std ratio: {np.nanmedian(comparison_stats['std_ratio']):.4f}")

# Save comparison statistics to CSV for further analysis
comparison_stats.to_csv('../data/engineered_features_comparison_stats.csv', index=True)

## Statistical Tests for Distribution Similarity

Beyond basic statistics, we can use statistical tests to evaluate whether the distributions are significantly different.

In [ ]:
# Statistical tests for distribution comparison
print("Running statistical tests for distribution similarity...")

# Initialize results list
test_results = []

# For each feature, perform KS test and t-test
for col in train_features_df.columns:
    # Get non-NaN values for both datasets
    train_vals = train_features_df[col].dropna().values
    test_vals = test_features_df[col].dropna().values
    
    # Skip if insufficient data
    if len(train_vals) < 2 or len(test_vals) < 2:
        continue
    
    # Kolmogorov-Smirnov test (compares distributions)
    ks_stat, ks_pvalue = stats.ks_2samp(train_vals, test_vals)
    
    # T-test for difference in means (Welch's t-test for unequal variances)
    t_stat, t_pvalue = stats.ttest_ind(train_vals, test_vals, equal_var=False)
    
    # Calculate Cohen's d (effect size)
    pooled_std = np.sqrt(((len(train_vals)-1)*np.var(train_vals, ddof=1) + (len(test_vals)-1)*np.var(test_vals, ddof=1)) / (len(train_vals) + len(test_vals) - 2))
    if pooled_std > 0:
        cohens_d = (np.mean(test_vals) - np.mean(train_vals)) / pooled_std
    else:
        cohens_d = 0
    
    # Store results
    test_results.append({
        'feature': col,
        'ks_statistic': ks_stat,
        'ks_pvalue': ks_pvalue,
        't_statistic': t_stat,
        't_pvalue': t_pvalue,
        'cohens_d': cohens_d,
        'train_mean': np.mean(train_vals),
        'test_mean': np.mean(test_vals),
        'train_std': np.std(train_vals, ddof=1),
        'test_std': np.std(test_vals, ddof=1)
    })

# Convert to DataFrame
results_df = pd.DataFrame(test_results)

# Summary of test results
print(f"\nKolmogorov-Smirnov Test Results:")
print(f"  Features with p < 0.05 (significantly different distributions): {(results_df['ks_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median KS statistic: {results_df['ks_statistic'].median():.4f}")

print(f"\nT-test Results (Mean Differences):")
print(f"  Features with p < 0.05 (significantly different means): {(results_df['t_pvalue'] < 0.05).sum()} out of {len(results_df)}")
print(f"  Median |Cohen's d|: {np.abs(results_df['cohens_d']).median():.4f}")

print(f"\nEffect Size Interpretation (Cohen's d):")
small_effect = (np.abs(results_df['cohens_d']) < 0.2).sum()
medium_effect = ((np.abs(results_df['cohens_d']) >= 0.2) & (np.abs(results_df['cohens_d']) < 0.5)).sum()
large_effect = (np.abs(results_df['cohens_d']) >= 0.5).sum()
print(f"  Small effect (<0.2): {small_effect} features")
print(f"  Medium effect (0.2-0.5): {medium_effect} features")
print(f"  Large effect (>0.5): {large_effect} features")

print(f"\nDetailed results (first 10 features):")
display(results_df[['feature', 'ks_pvalue', 't_pvalue', 'cohens_d']].head(10))


## Population Stability Index (PSI) Analysis

Population Stability Index (PSI) is a metric commonly used in machine learning to detect shifts in population distributions. It's particularly useful for monitoring data drift.

PSI Formula: PSI = Σ((% Actual - % Expected) * ln(% Actual / % Expected))

General guidelines:
- PSI < 0.1: No significant change
- 0.1 ≤ PSI < 0.2: Moderate change
- PSI ≥ 0.2: Significant change (potential data drift)


In [ ]:
# Calculate Population Stability Index (PSI) for each feature
print("Calculating Population Stability Index (PSI)...")

def calculate_psi(expected, actual, buckets=10):
    """Calculate Population Stability Index (PSI)."""
    # Remove any infinite or NaN values
    expected = expected[np.isfinite(expected)]
    actual = actual[np.isfinite(actual)]
    
    if len(expected) == 0 or len(actual) == 0:
        return np.nan
    
    # Create bins based on the expected distribution
    _, bin_edges = np.histogram(expected, bins=buckets)
    
    # Calculate percentages in each bucket
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # Convert to percentages
    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)
    
    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10
    expected_perc = np.maximum(expected_perc, epsilon)
    actual_perc = np.maximum(actual_perc, epsilon)
    
    # Calculate PSI
    psi = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi

# Calculate PSI for each feature
psi_results = []

for col in train_features_df.columns:
    # Get non-NaN values
    train_vals = train_features_df[col].dropna().values
    test_vals = test_features_df[col].dropna().values
    
    if len(train_vals) > 0 and len(test_vals) > 0:
        psi = calculate_psi(train_vals, test_vals, buckets=10)
        psi_results.append({
            'feature': col,
            'psi': psi
        })
    else:
        psi_results.append({
            'feature': col,
            'psi': np.nan
        })

# Convert to DataFrame and sort by PSI (descending)
psi_df = pd.DataFrame(psi_results)
psi_df = psi_df.sort_values('psi', ascending=False)

# Summary of PSI results
print(f"\nPopulation Stability Index (PSI) Results:")
print(f"  Features with PSI < 0.1 (no significant change): {(psi_df['psi'] < 0.1).sum()} out of {len(psi_df)}")
print(f"  Features with 0.1 ≤ PSI < 0.2 (moderate change): {((psi_df['psi'] >= 0.1) & (psi_df['psi'] < 0.2)).sum()} out of {len(psi_df)}")
print(f"  Features with PSI ≥ 0.2 (significant change): {(psi_df['psi'] >= 0.2).sum()} out of {len(psi_df)}")
print(f"  Median PSI: {psi_df['psi'].median():.4f}")
print(f"  Mean PSI: {psi_df['psi'].mean():.4f}")

print(f"\nTop 10 features with highest PSI:")
display(psi_df.head(10))

print(f"\nBottom 20 features with highest PSI:")
display(psi_df.tail(20))



## Visualization of Distributions - All features

Let's visualize the distributions of ALL engineered features to better understand the differences between train and test datasets, and save the plots to a PDF file.


In [ ]:
# Create a PDF with distribution plots for all features
print("Creating PDF with distribution plots for all features...")

# Create PDF file in data directory
pdf_path = '../data/feature_distributions.pdf'
pdf = PdfPages(pdf_path)

# Get all feature names sorted by PSI (descending) - highest drift first
all_features = psi_df.sort_values('psi', ascending=False)['feature'].tolist()

# Configuration for plots per page
plots_per_page = 28  # 4 rows x 7 columns = 28 plots per page
n_features = len(all_features)
n_pages = (n_features + plots_per_page - 1) // plots_per_page  # Ceiling division

print(f"Plotting {n_features} features across {n_pages} pages ({plots_per_page} plots per page)")

# Create plots for each page
for page_num in range(n_pages):
    # Determine which features to plot on this page
    start_idx = page_num * plots_per_page
    end_idx = min(start_idx + plots_per_page, n_features)
    features_on_page = all_features[start_idx:end_idx]

    # Calculate grid dimensions
    n_cols = 7
    n_rows = (len(features_on_page) + n_cols - 1) // n_cols  # Ceiling division

    # Create subplot grid
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    # Flatten axes for easy iteration
    axes_flat = axes.flatten()

    # Plot each feature
    for idx, feature in enumerate(features_on_page):
        # Get data
        train_data = train_features_df[feature].dropna()
        test_data = test_features_df[feature].dropna()

        # Create histogram
        ax = axes_flat[idx]
        n_bins = 30  # Reduced bins for cleaner plots with many subplots

        # Plot histograms
        alpha = 0.7
        ax.hist(train_data, bins=n_bins, alpha=alpha, label='Train', density=True, color='blue', edgecolor='none')
        ax.hist(test_data, bins=n_bins, alpha=alpha, label='Test', density=True, color='orange', edgecolor='none')

        # Add vertical lines for means
        ax.axvline(train_data.mean(), color='blue', linestyle='-', linewidth=1.5, alpha=0.8)
        ax.axvline(test_data.mean(), color='orange', linestyle='-', linewidth=1.5, alpha=0.8)

        # Get PSI value for this feature
        psi_value = psi_df[psi_df["feature"]==feature]["psi"].values[0]

        # Formatting
        ax.set_title(f'{feature}\nPSI: {psi_value:.3f}', fontsize=9)
        ax.set_xlabel('Value', fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    # Hide unused subplots on this page
    for idx in range(len(features_on_page), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Add title to the figure
    fig.suptitle(f'Feature Distributions: Train vs Test (Page {page_num+1}/{n_pages})',
                 fontsize=14, y=0.98)

    # Adjust layout and save to PDF
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

# Close the PDF
pdf.close()

print(f"✓ Saved feature distribution plots to: {os.path.abspath(pdf_path)}")
print(f"  Total pages: {n_pages}")
print(f"  Features per page: {plots_per_page}")
print(f"  Total features plotted: {n_features}")

# Also create a summary plot of PSI values and save it separately
print("\nCreating PSI summary plot...")
plt.figure(figsize=(14, 7))
psi_sorted = psi_df.sort_values('psi', ascending=False)
# Color code by PSI value
colors = []
for x in psi_sorted['psi']:
    if x >= 0.2:
        colors.append('red')
    elif x >= 0.1:
        colors.append('orange')
    else:
        colors.append('green')

bars = plt.bar(range(len(psi_sorted)), psi_sorted['psi'], color=colors, alpha=0.7, edgecolor='none')
plt.axhline(y=0.1, color='orange', linestyle='--', alpha=0.7, linewidth=1.5, label='Moderate change (PSI=0.1)')
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='Significant change (PSI=0.2)')
plt.xlabel('Features (ranked by PSI - highest drift first)', fontsize=12)
plt.ylabel('Population Stability Index (PSI)', fontsize=12)
plt.title('Population Stability Index (PSI) for All Engineered Features\n'
          'Red: Significant drift (≥0.2) | Orange: Moderate drift (0.1-0.2) | Green: Minimal drift (<0.1)',
          fontsize=14, pad=20)
plt.legend(loc='upper right')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()

# Save PSI summary plot
psi_plot_path = '../data/psi_summary_plot.png'
plt.savefig(psi_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Saved PSI summary plot to: {os.path.abspath(psi_plot_path)}")

# Print interpretation guide
print("\nPSI Interpretation Guide:")
print("- PSI < 0.1: No significant change")
print("- 0.1 ≤ PSI < 0.2: Moderate change")
print("- PSI ≥ 0.2: Significant change - indicates potential data drift")

## Summary and Conclusions

Based on the analysis above, we can draw the following conclusions about the similarity between the training and test datasets after feature engineering:

### Key Findings:

1. **Basic Statistics**: The mean and standard deviation differences between train and test datasets for engineered features indicate how much the central tendency and spread have shifted.

2. **Statistical Tests**:
   - The Kolmogorov-Smirnov test evaluates whether the overall distributions are similar
   - The t-test evaluates whether the means are significantly different
   - Cohen's d measures the effect size of any differences

3. **Population Stability Index (PSI)**:
   - This metric is particularly useful for detecting data drift in machine learning applications
   - PSI < 0.1 indicates no significant change
   - 0.1 ≤ PSI < 0.2 indicates moderate change
   - PSI ≥ 0.2 indicates significant change that may affect model performance

### Recommendations:

Based on the PSI results:
- If most features have PSI < 0.1: The train and test datasets are highly similar after feature engineering - good news for model generalization!
- If some features have 0.1 ≤ PSI < 0.2: Moderate changes exist but may not severely impact model performance
- If many features have PSI ≥ 0.2: Significant drift detected - consider investigating data collection processes or retraining models

### Next Steps:

1. Run this notebook to get the actual numerical results
2. Focus on the PSI analysis for practical guidance on data drift after feature engineering
3. Investigate any features with high PSI values to understand why they differ
4. Consider whether additional feature engineering or preprocessing adjustments are needed
5. If significant drift is found, you may need to retrain your model with more recent data or use domain adaptation techniques

**Note**: The -9999 values in the test dataset have been properly treated as NaN throughout this analysis, ensuring they don't distort the statistical comparisons.
